# Cuando el camino más corto no es el mejor
**Fundamentos de Inteligencia Artificial — Grupo LITIID71**

Prototipo de búsqueda de rutas en una zona inundada: un agente debe llegar desde una zona afectada hasta un refugio, evitando zonas bloqueadas y considerando el riesgo de cada zona según su nivel de inundación.

| Integrante | Rol |
|---|---|
| Hector Partido Flores | Coordinación e integración |
| Ismael Alexander Hidalgo Aguilando | Terreno y visualización |
| Hector Martínez Sánchez | Búsqueda no informada |
| Gildardo Emmanuel González García | Búsqueda informada y comparación |
| Anthony Oziel Ramos López | Validación y pruebas |

## 0. Configuración
Importaciones y contrato común del proyecto (`src/modelo.py`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from src.modelo import *

print("Entorno listo")

## 1. Introducción y problema
_Responsable: Hector Partido Flores_

- Descripción del escenario de inundación.
- Diferencia entre **encontrar una ruta** y **encontrar una ruta conveniente**.
- Pregunta guía: si dos rutas llegan al mismo refugio, pero una es más corta y la otra atraviesa zonas de mayor riesgo, ¿cómo determina el sistema cuál es la mejor?

## 2. Modelo del terreno y función de costo
_Responsable: Ismael Alexander Hidalgo Aguilando_

- Representación del terreno como cuadrícula de niveles de inundación (0 libre, 1 leve, 2 media, 3 bloqueada).
- Movimiento en 4 direcciones.
- Costo de entrar a una celda: `1 + λ · riesgo(nivel)`.
- Justificación de los valores de riesgo elegidos.
- Escenarios de prueba.

## 2. Modelo del terreno y función de costo
_Responsable: Ismael Alexander Hidalgo Aguilando_

### 2.1 Representación del terreno
El terreno se representa como una **cuadrícula** (matriz de NumPy) donde cada celda es una zona y guarda su nivel de inundación.

Se eligió la cuadrícula porque:
- Permite construir y modificar escenarios fácilmente: bloquear una zona o cambiar su riesgo es cambiar un rectángulo de la matriz (`colocar_zona`).
- Se puede dibujar directamente como imagen, con un color por nivel.
- Cada celda tiene como máximo 4 vecinos, así que el problema es fácil de seguir a mano al explicar los algoritmos.

Un grafo general (calles e intersecciones) sería más realista, pero para comparar estrategias de búsqueda la cuadrícula basta y hace los resultados más fáciles de verificar.

### 2.2 Movimientos
El agente se mueve en **4 direcciones** (arriba, abajo, izquierda, derecha), sin diagonales:
- Todos los pasos miden lo mismo (1), así que contar pasos es medir distancia.
- Se evita "cortar esquinas" pasando en diagonal entre dos celdas bloqueadas.
- La distancia Manhattan nunca sobreestima los pasos necesarios, lo que la hace una heurística admisible para A\*.

### 2.3 Niveles de inundación y riesgo

| Nivel | Significado | Transitable | Riesgo |
|---|---|---|---|
| 0 | Seco | Sí | 0 |
| 1 | Inundación leve | Sí | 1 |
| 2 | Inundación media | Sí | 3 |
| 3 | Inundación severa | **No** | — |

- **El riesgo no crece de forma lineal (0, 1, 3).** Pasar de agua leve a agua media es mucho más peligroso que pasar de seco a leve: hay más profundidad y corriente. Por eso una celda media pesa el triple que una leve.
- **La inundación severa es una restricción, no un costo alto.** Se trata como zona bloqueada: ningún algoritmo puede pisarla, sin importar cuánto ahorre. Así se distinguen dos tipos de condición:
  - **restricciones:** lo que nunca se permite;
  - **penalizaciones:** lo que se permite, pero cuesta más.

### 2.4 Función de costo
El costo de **entrar** a una celda es:

$$\text{costo de paso} = 1 + \lambda \cdot \text{riesgo}(\text{celda})$$

y el costo total de una ruta es:

$$\text{costo de la ruta} = \text{pasos} + \lambda \cdot \text{riesgo acumulado}$$

- El **1** representa la distancia. Garantiza que todo paso cueste al menos 1, así que ninguna ruta es "gratis".
- **λ** indica cuánto pesa la seguridad frente a la distancia:
  - con **λ = 0** solo importan los pasos y se obtiene la ruta más corta;
  - al **aumentar λ**, cruzar agua se vuelve caro y el algoritmo prefiere rodearla.

**Respuesta a la pregunta guía:** si dos rutas llegan al mismo refugio, el sistema no elige la más corta ni la más segura por separado. Calcula el costo total de cada una con esta fórmula y elige la de **menor costo**. La ruta corta gana solo si el riesgo que ahorra la ruta segura no compensa los pasos extra.

In [ ]:
from src.terreno import crear_terreno, colocar_zona
from src.escenarios import escenario_atajo_inundado, escenario_dos_refugios
from src.visualizacion import dibujar_terreno, comparar_rutas, leyenda, NOMBRES_NIVEL

# Tabla del modelo: riesgo y costo de entrar a cada tipo de celda con λ = 1
pd.DataFrame([
    {
        "Nivel": NOMBRES_NIVEL[n],
        "Transitable": "Sí" if n != BLOQUEADA else "No",
        "Riesgo": RIESGO_POR_NIVEL.get(n, "—"),
        "Costo de paso (λ=1)": 1 + RIESGO_POR_NIVEL[n] if n in RIESGO_POR_NIVEL else "—",
    }
    for n in (LIBRE, LEVE, MEDIA, BLOQUEADA)
])

### 2.5 ¿Cuántos pasos extra vale la pena caminar para evitar el agua?
Como cada paso cuesta 1, el término `λ · riesgo` se puede leer como **pasos extra**. Evitar una celda inundada vale la pena si el rodeo agrega menos pasos que `λ · riesgo` de esa celda.

In [ ]:
equivalencias = []
for lam in (0, 0.5, 1, 2, 5):
    fila = {"λ": lam}
    for n in (LEVE, MEDIA):
        fila[f"{NOMBRES_NIVEL[n]}: costo de paso"] = 1 + lam * RIESGO_POR_NIVEL[n]
        fila[f"{NOMBRES_NIVEL[n]}: pasos extra aceptados"] = lam * RIESGO_POR_NIVEL[n]
    equivalencias.append(fila)

pd.DataFrame(equivalencias).set_index("λ")

**Cómo leer la tabla:**
- Con **λ = 0**, el agua no cuesta nada extra: el algoritmo nunca da un rodeo para evitarla.
- Con **λ = 1**, evitar una celda de inundación media justifica caminar hasta **3 pasos extra**, y una leve hasta **1 paso extra**.
- Con **λ = 5**, el sistema es muy conservador: acepta hasta 15 pasos extra para no pisar una celda media.

Así, λ es un parámetro que se puede ajustar según la situación. Por ejemplo, con una evacuación nocturna o con personas vulnerables convendría un λ alto. En la sección 5.1 se muestra con ejecuciones reales cómo cambia la ruta elegida al variar λ.

In [ ]:
terreno, inicio, refugios = escenario_atajo_inundado()
fig, ax = plt.subplots(figsize=(7, 6), layout="constrained")
dibujar_terreno(terreno, inicio, refugios, titulo="Escenario principal: atajo inundado", ax=ax)
leyenda(fig, con_ruta=False)
plt.show()

## 3. Estrategias de búsqueda

Todas las estrategias reciben lo mismo (`terreno, inicio, refugios, lam`) y devuelven un `Resultado` con la misma estructura. Para que la comparación sea justa, todas miden el trabajo con los mismos criterios:

| Métrica | Qué mide |
|---|---|
| **Nodos generados** | Cuántas celdas se agregaron a la frontera (incluye el inicio). |
| **Nodos expandidos** | Cuántas celdas se sacaron de la frontera y se revisaron sus vecinos. Es la medida principal de trabajo. |
| **Frontera máxima** | El mayor número de celdas pendientes al mismo tiempo. Indica cuánta memoria usó la búsqueda. |
| **Tiempo** | Milisegundos medidos con `time.perf_counter()`. Varía entre ejecuciones y equipos. |

En todas, la prueba de meta se hace **al sacar** la celda de la frontera, no al agregarla. Así las métricas son comparables entre algoritmos.

### 3.1 Búsqueda no informada (BFS, DFS, UCS)
_Responsable: Hector Martínez Sánchez_

Las estrategias no informadas no saben dónde está el refugio: solo exploran a partir del inicio. La diferencia entre ellas es **qué celda de la frontera eligen expandir primero**.

| Algoritmo | Estructura de la frontera | Expande primero | ¿Usa el riesgo para decidir? | Garantiza |
|---|---|---|---|---|
| **BFS** (anchura) | Cola (FIFO) | La celda que lleva más tiempo esperando | No | La ruta con **menos pasos** |
| **DFS** (profundidad) | Pila (LIFO) | La celda agregada más recientemente | No | Solo **encontrar una ruta**, si existe |
| **UCS** (costo uniforme) | Cola de prioridad por costo acumulado `g` | La celda con menor costo acumulado | **Sí** | La ruta de **menor costo** |

- **BFS** avanza por "capas": primero todas las celdas a 1 paso, luego a 2, y así sucesivamente. Por eso la primera vez que llega al refugio lo hace con el mínimo de pasos, aunque cruce agua.
- **DFS** sigue un camino hasta el fondo antes de regresar. Encuentra una ruta, pero puede ser muy larga y dar muchas vueltas. Se incluye como contraste: **encontrar una ruta no es lo mismo que encontrar una ruta conveniente.**
- **UCS** siempre expande la celda más barata hasta el momento, con costo `1 + λ·riesgo` por paso. Es la única de las tres que considera el riesgo al decidir.

En BFS y DFS, λ no cambia la ruta: solo se usa para **reportar** cuánto cuesta la ruta que encontraron.

### 3.2 Búsqueda informada (Greedy Best-First, A*)
_Responsable: Gildardo Emmanuel González García_

- Heurística utilizada y por qué es admisible.

## 4. Visualización del terreno y las rutas
_Responsable: Ismael Alexander Hidalgo Aguilando_

- Mapa con colores por nivel de inundación.
- Rutas de cada algoritmo dibujadas sobre el terreno.

In [ ]:
from src.busqueda_no_informada import bfs, dfs, ucs
from src.busqueda_informada import greedy, a_estrella

ALGORITMOS = [bfs, dfs, ucs, greedy, a_estrella]

resultados = [alg(terreno, inicio, refugios, lam=1.0) for alg in ALGORITMOS]
fig = comparar_rutas(terreno, inicio, refugios, resultados)
fig.suptitle("Escenario principal (λ = 1)", x=0.01, ha="left", fontsize=11)
plt.show()

In [ ]:
terreno_2, inicio_2, refugios_2 = escenario_dos_refugios()
resultados_2 = [alg(terreno_2, inicio_2, refugios_2, lam=1.0) for alg in ALGORITMOS]
fig = comparar_rutas(terreno_2, inicio_2, refugios_2, resultados_2)
fig.suptitle("Dos refugios: uno cercano tras el agua y otro lejano en seco (λ = 1)",
             x=0.01, ha="left", fontsize=11)
plt.show()

**Observaciones**

- En el escenario principal, BFS y Greedy cruzan la zona inundada: 11 pasos, pero costo 25. UCS y A* la rodean por el norte: 19 pasos, costo 19. La ruta más corta no es la más conveniente.
- DFS encuentra una ruta, pero es muy larga y atraviesa agua varias veces. Encontrar una ruta no es lo mismo que encontrar una ruta conveniente.
- En el escenario de dos refugios, BFS y Greedy eligen el refugio cercano cruzando el agua (costo 16). UCS y A* eligen el refugio lejano por terreno seco (costo 15).

## 5. Comparación de algoritmos
_Responsable: Gildardo Emmanuel González García_

Tabla con resultados de **ejecuciones reales**: costo, pasos, riesgo acumulado, nodos expandidos, nodos generados, frontera máxima y tiempo.

### 5.1 Efecto del peso del riesgo (λ)
Cómo cambia la ruta elegida al aumentar λ, y en qué valor deja de convenir la ruta más corta.

## 6. Pruebas
_Responsable: Anthony Oziel Ramos López_

Todas las rutas se comprueban con el validador (`src/validacion.py`).

### 6.1 Existe una ruta disponible hasta el refugio

### 6.2 Se bloquea una zona que formaba parte de la ruta

### 6.3 El refugio deja de ser accesible

### 6.4 Se modifica el riesgo de una región y se vuelve a buscar

### 6.5 El camino más corto deja de ser el más conveniente

### 6.6 Prueba adicional: detección de errores de implementación
En mapas aleatorios, A* y UCS deben obtener siempre el mismo costo óptimo. Si difieren, hay un error en la heurística o en la implementación.

## 7. Decisiones de diseño y conclusiones
_Responsable: Hector Partido Flores (con aportes de todos)_

- Por qué una cuadrícula y movimiento en 4 direcciones.
- Cómo se incorpora el riesgo en el costo.
- Respuesta a la pregunta guía, con base en los resultados de las secciones 5 y 6.
- Diferencias observadas entre estrategias.

## 8. Declaración de uso de herramientas de IA
_Responsable: Hector Partido Flores_

- Herramientas utilizadas y para qué.
- Cómo se verificó el código generado (pruebas, validador, revisión en Pull Requests).